# SALES ANALYTICS

In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import math
import matplotlib.pyplot as plt
from sqlalchemy import create_engine
from urllib.parse import quote_plus
import os
from dotenv import load_dotenv

# Load environment variables from .env file
load_dotenv()
# Database connection parameters
DB_USER = os.getenv('DB_USER')
DB_PASSWORD = os.getenv('DB_PASSWORD')
DB_HOST = os.getenv('DB_HOST')
DB_PORT = os.getenv('DB_PORT')
DB_NAME = os.getenv('DB_NAME')


In [2]:
# create a connection string
def create_connection_string(user, password, host, port, db_name):
    return f'mysql+mysqlconnector://{user}:{quote_plus(password)}@{host}:{port}/{db_name}'
# create a connection string
connection_string = create_connection_string(DB_USER, DB_PASSWORD, DB_HOST, DB_PORT, DB_NAME)
# create a database engine
engine = create_engine(connection_string)

In [3]:
#loading all tables into dataframes
df_customers = pd.read_sql("SELECT * FROM customers;", engine)
df_dates = pd.read_sql("SELECT * FROM date;", engine)
df_markets = pd.read_sql("SELECT * FROM markets;", engine)
df_products = pd.read_sql("SELECT * FROM products;", engine)
df_transactions = pd.read_sql("SELECT * FROM transactions;", engine)

In [4]:
# function to get table info
def get_table_info(df, table_name):
    print(f"Table: {table_name}")
    print("Table Info:", df.info())
    print("Shape:", df.shape)
    print("Columns:", df.columns.tolist())
    print("Data Types:\n", df.dtypes)
    print("Missing Values:\n", df.isnull().sum())
    print("Duplicates:\n", df.duplicated().sum())
    print("\nSample Data:\n", df.head())
    print("-" * 50)

## Customer Table

In [5]:
#customers table info
get_table_info(df_customers, "customers")

Table: customers
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 38 entries, 0 to 37
Data columns (total 3 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   customer_code  38 non-null     object
 1   custmer_name   38 non-null     object
 2   customer_type  38 non-null     object
dtypes: object(3)
memory usage: 1.0+ KB
Table Info: None
Shape: (38, 3)
Columns: ['customer_code', 'custmer_name', 'customer_type']
Data Types:
 customer_code    object
custmer_name     object
customer_type    object
dtype: object
Missing Values:
 customer_code    0
custmer_name     0
customer_type    0
dtype: int64
Duplicates:
 0

Sample Data:
   customer_code    custmer_name   customer_type
0        Cus001    Surge Stores  Brick & Mortar
1        Cus002    Nomad Stores  Brick & Mortar
2        Cus003    Excel Stores  Brick & Mortar
3        Cus004  Surface Stores  Brick & Mortar
4        Cus005  Premium Stores  Brick & Mortar
------------------------------

In [6]:
df_customers

,customer_code,custmer_name,customer_type
0,Cus001,Surge Stores,Brick & Mortar
1,Cus002,Nomad Stores,Brick & Mortar
2,Cus003,Excel Stores,Brick & Mortar
3,Cus004,Surface Stores,Brick & Mortar
4,Cus005,Premium Stores,Brick & Mortar
5,Cus006,Electricalsara Stores,Brick & Mortar
6,Cus007,Info Stores,Brick & Mortar
7,Cus008,Acclaimed Stores,Brick & Mortar
8,Cus009,Electricalsquipo Stores,Brick & Mortar
9,Cus010,Atlas Stores,Brick & Mortar


## Dates Table

In [7]:
get_table_info(df_dates, "date")

Table: date
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1126 entries, 0 to 1125
Data columns (total 5 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   date         1126 non-null   object
 1   cy_date      1126 non-null   object
 2   year         1126 non-null   int64 
 3   month_name   1126 non-null   object
 4   date_yy_mmm  1126 non-null   object
dtypes: int64(1), object(4)
memory usage: 44.1+ KB
Table Info: None
Shape: (1126, 5)
Columns: ['date', 'cy_date', 'year', 'month_name', 'date_yy_mmm']
Data Types:
 date           object
cy_date        object
year            int64
month_name     object
date_yy_mmm    object
dtype: object
Missing Values:
 date           0
cy_date        0
year           0
month_name     0
date_yy_mmm    0
dtype: int64
Duplicates:
 0

Sample Data:
          date     cy_date  year month_name date_yy_mmm
0  2017-06-01  2017-06-01  2017       June    17-Jun\r
1  2017-06-02  2017-06-01  2017       June    17-Jun

In [8]:
def extract_date_features(df, col, prefix=None):
    df[col] = pd.to_datetime(df[col])
    pre = prefix if prefix else col
    df[f'{pre}_day'] = df[col].dt.day
    df[f'{pre}_day_of_week'] = df[col].dt.dayofweek
    df[f'{pre}_day_name'] = df[col].dt.day_name()
    df[f'{pre}_month'] = df[col].dt.month
    df[f'{pre}_month_name'] = df[col].dt.month_name()
    df[f'{pre}_year'] = df[col].dt.year
    return df

# Extract features for both columns
df_dates = extract_date_features(df_dates, 'date')
df_dates = extract_date_features(df_dates, 'cy_date', prefix='cy')

# Reorder columns
ordered_cols = [
    'date', 'date_day', 'date_day_of_week', 'date_day_name', 'date_month', 'date_month_name', 'date_year',
    'cy_date', 'cy_day', 'cy_day_of_week', 'cy_day_name', 'cy_month', 'cy_month_name', 'cy_year'
]

df_dates = df_dates[ordered_cols]

In [9]:
df_dates

,date,date_day,date_day_of_week,date_day_name,date_month,date_month_name,date_year,cy_date,cy_day,cy_day_of_week,cy_day_name,cy_month,cy_month_name,cy_year
0,2017-06-01,1,3,Thursday,6,June,2017,2017-06-01,1,3,Thursday,6,June,2017
1,2017-06-02,2,4,Friday,6,June,2017,2017-06-01,1,3,Thursday,6,June,2017
2,2017-06-03,3,5,Saturday,6,June,2017,2017-06-01,1,3,Thursday,6,June,2017
3,2017-06-04,4,6,Sunday,6,June,2017,2017-06-01,1,3,Thursday,6,June,2017
4,2017-06-05,5,0,Monday,6,June,2017,2017-06-01,1,3,Thursday,6,June,2017
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1121,2020-06-26,26,4,Friday,6,June,2020,2020-06-01,1,0,Monday,6,June,2020
1122,2020-06-27,27,5,Saturday,6,June,2020,2020-06-01,1,0,Monday,6,June,2020
1123,2020-06-28,28,6,Sunday,6,June,2020,2020-06-01,1,0,Monday,6,June,2020
1124,2020-06-29,29,0,Monday,6,June,2020,2020-06-01,1,0,Monday,6,June,2020


In [10]:
get_table_info(df_markets, "markets")

Table: markets
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 17 entries, 0 to 16
Data columns (total 3 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   markets_code  17 non-null     object
 1   markets_name  17 non-null     object
 2   zone          17 non-null     object
dtypes: object(3)
memory usage: 540.0+ bytes
Table Info: None
Shape: (17, 3)
Columns: ['markets_code', 'markets_name', 'zone']
Data Types:
 markets_code    object
markets_name    object
zone            object
dtype: object
Missing Values:
 markets_code    0
markets_name    0
zone            0
dtype: int64
Duplicates:
 0

Sample Data:
   markets_code markets_name     zone
0      Mark001      Chennai    South
1      Mark002       Mumbai  Central
2      Mark003    Ahmedabad    North
3      Mark004    Delhi NCR    North
4      Mark005       Kanpur    North
--------------------------------------------------


In [11]:
# Check if zone has truly empty strings
df_markets[df_markets['zone'].str.strip() == ""]
df_markets['zone'] = df_markets['zone'].replace(r'^\s*$', np.nan, regex=True)
# Replace empty strings or whitespace-only strings with NaN
df_markets['zone'] = df_markets['zone'].replace(r'^\s*$', np.nan, regex=True)
df_markets['zone'] = df_markets['zone'].fillna('Overseas')

In [12]:
df_markets

,markets_code,markets_name,zone
0,Mark001,Chennai,South
1,Mark002,Mumbai,Central
2,Mark003,Ahmedabad,North
3,Mark004,Delhi NCR,North
4,Mark005,Kanpur,North
5,Mark006,Bengaluru,South
6,Mark007,Bhopal,Central
7,Mark008,Lucknow,North
8,Mark009,Patna,North
9,Mark010,Kochi,South


In [13]:
get_table_info(df_products, "products")

Table: products
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 279 entries, 0 to 278
Data columns (total 2 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   product_code  279 non-null    object
 1   product_type  279 non-null    object
dtypes: object(2)
memory usage: 4.5+ KB
Table Info: None
Shape: (279, 2)
Columns: ['product_code', 'product_type']
Data Types:
 product_code    object
product_type    object
dtype: object
Missing Values:
 product_code    0
product_type    0
dtype: int64
Duplicates:
 0

Sample Data:
   product_code product_type
0      Prod001  Own Brand\r
1      Prod002  Own Brand\r
2      Prod003  Own Brand\r
3      Prod004  Own Brand\r
4      Prod005  Own Brand\r
--------------------------------------------------


In [14]:
df_products

,product_code,product_type
0,Prod001,Own Brand\r
1,Prod002,Own Brand\r
2,Prod003,Own Brand\r
3,Prod004,Own Brand\r
4,Prod005,Own Brand\r
...,...,...
274,Prod275,Own Brand\r
275,Prod276,Own Brand\r
276,Prod277,Own Brand\r
277,Prod278,Distribution\r


## Transactions

In [15]:
get_table_info(df_transactions, "transactions")

Table: transactions
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 150283 entries, 0 to 150282
Data columns (total 7 columns):
 #   Column         Non-Null Count   Dtype  
---  ------         --------------   -----  
 0   product_code   150283 non-null  object 
 1   customer_code  150283 non-null  object 
 2   market_code    150283 non-null  object 
 3   order_date     150283 non-null  object 
 4   sales_qty      150283 non-null  int64  
 5   sales_amount   150283 non-null  float64
 6   currency       150283 non-null  object 
dtypes: float64(1), int64(1), object(5)
memory usage: 8.0+ MB
Table Info: None
Shape: (150283, 7)
Columns: ['product_code', 'customer_code', 'market_code', 'order_date', 'sales_qty', 'sales_amount', 'currency']
Data Types:
 product_code      object
customer_code     object
market_code       object
order_date        object
sales_qty          int64
sales_amount     float64
currency          object
dtype: object
Missing Values:
 product_code     0
customer_code   

In [16]:
# cleaning
df_transactions.order_date = pd.to_datetime(df_transactions.order_date)
df_transactions = df_transactions[df_transactions['sales_amount'] > 0]

#conversion rates
usd_to_inr = 82.0
df_transactions.loc[df_transactions['currency'] == 'USD', 'sales_amount'] *= usd_to_inr
df_transactions['currency'] = 'INR'

In [17]:
df_transactions.head()

,product_code,customer_code,market_code,order_date,sales_qty,sales_amount,currency
0,Prod001,Cus001,Mark001,2017-10-10,100,41241.0,INR
2,Prod002,Cus003,Mark003,2018-04-06,1,875.0,INR
3,Prod002,Cus003,Mark003,2018-04-11,1,583.0,INR
4,Prod002,Cus004,Mark003,2018-06-18,6,7176.0,INR
5,Prod003,Cus005,Mark004,2017-11-20,59,41000.0,INR
